In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

!pip install -q -U transformers accelerate bitsandbytes huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.1/721.1 kB 53.4 MB/s eta 0:00:00


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from huggingface_hub import login
from google.colab import userdata

login(token=userdata.get('HF_token'))

MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
MODEL_NAME = "llama_3.1_8b"
N_LAYERS = 32

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, device_map="auto",
)
model.eval()
torch.cuda.empty_cache()

print(f"Model loaded: {MODEL_ID}")
print(f"GPU memory: {torch.cuda.memory_allocated()/1e9:.2f} GB")

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

Model loaded: meta-llama/Llama-3.1-8B-Instruct
GPU memory: 6.03 GB


In [ ]:
import os, json, re, subprocess
import numpy as np

def query(system, user, max_new_tokens=200):
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(model.device)
    output = model.generate(
        **inputs, max_new_tokens=max_new_tokens, do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(output[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

# Clone RuleArena
if not os.path.exists("RuleArena"):
    subprocess.run(["git", "clone", "-q", "https://github.com/SkyRiver-2000/RuleArena.git"], check=True)

os.chdir("/content/RuleArena/airline")
exec(open("compute_answer.py").read().split("if __name__")[0])
check_base_tables = load_checking_fee()

with open("reference_rules.txt") as f:
    REFERENCE_RULES = f.read()

problems = []
with open("synthesized_problems/comp_0.jsonl") as f:
    for line in f:
        problems.append(json.loads(line))

def compute_gt(info):
    total, gt_info = compute_answer(
        base_price=info["base_price"],
        direction=info["direction"],
        routine=info["routine"],
        customer_class=info["customer_class"],
        bag_list=[{"id": 0, "name": "ticket", "size": [0,0,0], "weight": 0}] + info["bag_list"],
        check_base_tables=check_base_tables,
    )
    return total, gt_info

print(f"Loaded {len(problems)} problems.")
gt, gt_info = compute_gt(problems[0]["info"])
print(f"Problem 0 GT: ${gt}")

Loaded 100 problems.
Problem 0 GT: $1445


In [ ]:
def get_oversize_bracket(dim_sum):
    if dim_sum <= 62:
        return "NOT OVERSIZE"
    elif dim_sum <= 65:
        return "62-65 inches"
    else:
        return "65-115 inches"

def get_overweight_bracket(weight):
    if weight <= 50:
        return "NOT OVERWEIGHT"
    elif weight <= 53:
        return "50-53 lbs"
    elif weight <= 70:
        return "53-70 lbs"
    else:
        return "over 70 lbs"

def generate_steps(problem, gt_info):
    """
    Generate all elementary and boundary steps for a problem.
    Returns list of steps with: type, question, context, correct_answer
    """
    info = problem["info"]
    steps = []

    for i, bag in enumerate(info["bag_list"]):
        bag_id = bag["id"]
        dims = bag["size"]
        weight = bag["weight"]
        dim_sum = sum(dims)
        oversize_bracket = get_oversize_bracket(dim_sum)
        overweight_bracket = get_overweight_bracket(weight)

        correct_oversize_fee = gt_info["oversize"][i]
        correct_overweight_fee = gt_info["overweight"][i]
        correct_base_fee = gt_info["base"][i]

        # ── ELEMENTARY STEPS ─────────────────────────────────

        # E1: Read dimension sum (given the sum, just confirm it)
        steps.append({
            "step_id": f"bag{bag_id}_E1",
            "bag_id": bag_id,
            "type": "elementary",
            "description": "Read dimension sum",
            "system": "You are a baggage analyst. Answer with just the number.",
            "user": (
                f"A bag has dimensions {dims[0]} x {dims[1]} x {dims[2]} inches.\n"
                f"What is the total dimension sum (length + width + height)?\n"
                f"Answer with just the number in inches."
            ),
            "correct_answer": str(dim_sum),
        })

        # E2: Compare to oversize threshold
        expected_e2 = "OVERSIZE" if dim_sum > 62 else "NOT OVERSIZE"
        steps.append({
            "step_id": f"bag{bag_id}_E2",
            "bag_id": bag_id,
            "type": "elementary",
            "description": "Compare dimensions to 62-inch threshold",
            "system": "You are a baggage analyst. Answer with exactly OVERSIZE or NOT OVERSIZE.",
            "user": (
                f"A bag has total dimensions of {dim_sum} inches.\n"
                f"The oversize threshold is 62 inches.\n"
                f"Is this bag OVERSIZE or NOT OVERSIZE?\n"
                f"Answer with exactly one of: OVERSIZE or NOT OVERSIZE"
            ),
            "correct_answer": expected_e2,
        })

        # E3: Compare to overweight threshold
        expected_e3 = "OVERWEIGHT" if weight > 50 else "NOT OVERWEIGHT"
        steps.append({
            "step_id": f"bag{bag_id}_E3",
            "bag_id": bag_id,
            "type": "elementary",
            "description": "Compare weight to 50-lb threshold",
            "system": "You are a baggage analyst. Answer with exactly OVERWEIGHT or NOT OVERWEIGHT.",
            "user": (
                f"A bag weighs {weight} lbs.\n"
                f"The overweight threshold is 50 lbs.\n"
                f"Is this bag OVERWEIGHT or NOT OVERWEIGHT?\n"
                f"Answer with exactly one of: OVERWEIGHT or NOT OVERWEIGHT"
            ),
            "correct_answer": expected_e3,
        })

        # ── BOUNDARY STEPS ───────────────────────────────────

        # B1: Look up oversize fee (given bracket, must look up fee)
        if dim_sum > 62:
            steps.append({
                "step_id": f"bag{bag_id}_B1",
                "bag_id": bag_id,
                "type": "boundary",
                "description": "Look up oversize fee from table",
                "system": "You are an airline fee calculator. Answer with just the dollar amount.",
                "user": (
                    f"Route: {gt_info['routine']} domestic\n"
                    f"Passenger class: {gt_info['customer_class']}\n"
                    f"Bag {bag_id}: total dimensions = {dim_sum} inches "
                    f"(bracket: {oversize_bracket})\n\n"
                    f"FEE TABLE (Oversize section):\n"
                    f"{REFERENCE_RULES}\n\n"
                    f"What is the oversize fee for this bag?\n"
                    f"Answer with just the dollar amount, e.g. $30 or $200"
                ),
                "correct_answer": f"${correct_oversize_fee}",
            })

        # B2: Look up overweight fee (given bracket, must look up fee)
        if weight > 50:
            steps.append({
                "step_id": f"bag{bag_id}_B2",
                "bag_id": bag_id,
                "type": "boundary",
                "description": "Look up overweight fee from table",
                "system": "You are an airline fee calculator. Answer with just the dollar amount.",
                "user": (
                    f"Route: {gt_info['routine']} domestic\n"
                    f"Passenger class: {gt_info['customer_class']}\n"
                    f"Bag {bag_id}: weight = {weight} lbs "
                    f"(bracket: {overweight_bracket})\n\n"
                    f"FEE TABLE (Overweight section):\n"
                    f"{REFERENCE_RULES}\n\n"
                    f"What is the overweight fee for this bag?\n"
                    f"Answer with just the dollar amount, e.g. $30 or $100"
                ),
                "correct_answer": f"${correct_overweight_fee}",
            })

        # B3: Look up base bag fee (given bag number, must look up fee)
        steps.append({
            "step_id": f"bag{bag_id}_B3",
            "bag_id": bag_id,
            "type": "boundary",
            "description": "Look up base bag fee from table",
            "system": "You are an airline fee calculator. Answer with just the dollar amount.",
            "user": (
                f"Route: {gt_info['routine']} domestic\n"
                f"Passenger class: {gt_info['customer_class']}\n"
                f"This is bag number {bag_id} (the {['1st','2nd','3rd','4th','5th'][bag_id-1]} checked bag).\n\n"
                f"FEE TABLE:\n"
                f"{REFERENCE_RULES}\n\n"
                f"What is the base checked bag fee for this bag?\n"
                f"Answer with just the dollar amount, e.g. $40 or $45"
            ),
            "correct_answer": f"${correct_base_fee}",
        })

    return steps

def extract_answer(response, step_type):
    """Extract answer from model response"""
    response = response.strip()
    if step_type == "elementary":
        # For E1: extract number
        nums = re.findall(r'\b\d+\b', response)
        if nums:
            return nums[0]
        # For E2/E3: extract OVERSIZE/NOT OVERSIZE/OVERWEIGHT/NOT OVERWEIGHT
        for keyword in ["NOT OVERSIZE", "NOT OVERWEIGHT", "OVERSIZE", "OVERWEIGHT"]:
            if keyword in response.upper():
                return keyword
        return response.split()[0] if response else ""
    else:
        # For boundary: extract dollar amount
        match = re.search(r'\$(\d+)', response)
        if match:
            return f"${match.group(1)}"
        nums = re.findall(r'\b\d+\b', response)
        if nums:
            return f"${nums[0]}"
        return response.strip()

# Test on problem 0
steps = generate_steps(problems[0], compute_gt(problems[0]["info"])[1])
print(f"Problem 0: {len(steps)} steps generated")
elem = [s for s in steps if s['type'] == 'elementary']
bound = [s for s in steps if s['type'] == 'boundary']
print(f"  Elementary: {len(elem)}")
print(f"  Boundary: {len(bound)}")
print()
print("Step IDs:")
for s in steps:
    print(f"  {s['step_id']} ({s['type']}): {s['description']} → correct={s['correct_answer']}")

Problem 0: 28 steps generated
  Elementary: 15
  Boundary: 13

Step IDs:
  bag1_E1 (elementary): Read dimension sum → correct=41
  bag1_E2 (elementary): Compare dimensions to 62-inch threshold → correct=NOT OVERSIZE
  bag1_E3 (elementary): Compare weight to 50-lb threshold → correct=NOT OVERWEIGHT
  bag1_B3 (boundary): Look up base bag fee from table → correct=$40
  bag2_E1 (elementary): Read dimension sum → correct=86
  bag2_E2 (elementary): Compare dimensions to 62-inch threshold → correct=OVERSIZE
  bag2_E3 (elementary): Compare weight to 50-lb threshold → correct=OVERWEIGHT
  bag2_B1 (boundary): Look up oversize fee from table → correct=$200
  bag2_B2 (boundary): Look up overweight fee from table → correct=$100
  bag2_B3 (boundary): Look up base bag fee from table → correct=$45
  bag3_E1 (elementary): Read dimension sum → correct=64
  bag3_E2 (elementary): Compare dimensions to 62-inch threshold → correct=OVERSIZE
  bag3_E3 (elementary): Compare weight to 50-lb threshold → correct=

In [ ]:
def run_steps(problem, gt_info, steps):
    results = []
    for step in steps:
        response = query(step["system"], step["user"], max_new_tokens=100)
        predicted = extract_answer(response, step["type"])
        correct = predicted.strip().upper() == step["correct_answer"].strip().upper()
        results.append({
            "step_id": step["step_id"],
            "type": step["type"],
            "description": step["description"],
            "correct_answer": step["correct_answer"],
            "predicted": predicted,
            "correct": correct,
            "response": response,
        })
        print(f"{step['step_id']} ({step['type']}): expected={step['correct_answer']} predicted={predicted} {'✓' if correct else '✗'}")
    return results

print("=== PROBLEM 0 — ALL STEPS ===\n")
gt, gt_info = compute_gt(problems[0]["info"])
steps = generate_steps(problems[0], gt_info)
results_p0 = run_steps(problems[0], gt_info, steps)

elem_correct = sum(1 for r in results_p0 if r["type"] == "elementary" and r["correct"])
elem_total = sum(1 for r in results_p0 if r["type"] == "elementary")
bound_correct = sum(1 for r in results_p0 if r["type"] == "boundary" and r["correct"])
bound_total = sum(1 for r in results_p0 if r["type"] == "boundary")

print(f"\n=== SUMMARY ===")
print(f"Elementary: {elem_correct}/{elem_total} correct ({elem_correct/elem_total*100:.1f}%)")
print(f"Boundary:   {bound_correct}/{bound_total} correct ({bound_correct/bound_total*100:.1f}%)")

=== PROBLEM 0 — ALL STEPS ===



/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


bag1_E1 (elementary): expected=41 predicted=22 ✗
bag1_E2 (elementary): expected=NOT OVERSIZE predicted=NOT OVERSIZE ✓
bag1_E3 (elementary): expected=NOT OVERWEIGHT predicted=NOT OVERWEIGHT ✓
bag1_B3 (boundary): expected=$40 predicted=$40 ✓
bag2_E1 (elementary): expected=86 predicted=44 ✗
bag2_E2 (elementary): expected=OVERSIZE predicted=OVERSIZE ✓
bag2_E3 (elementary): expected=OVERWEIGHT predicted=OVERWEIGHT ✓
bag2_B1 (boundary): expected=$200 predicted=$200 ✓
bag2_B2 (boundary): expected=$100 predicted=$100 ✓
bag2_B3 (boundary): expected=$45 predicted=$45 ✓
bag3_E1 (elementary): expected=64 predicted=34 ✗
bag3_E2 (elementary): expected=OVERSIZE predicted=NOT OVERSIZE ✗
bag3_E3 (elementary): expected=OVERWEIGHT predicted=OVERWEIGHT ✓
bag3_B1 (boundary): expected=$30 predicted=$30 ✓
bag3_B2 (boundary): expected=$30 predicted=$30 ✓
bag3_B3 (boundary): expected=$150 predicted=$150 ✓
bag4_E1 (elementary): expected=76 predicted=38 ✗
bag4_E2 (elementary): expected=OVERSIZE predicted=OVERSIZ

In [ ]:
def generate_steps(problem, gt_info):
    info = problem["info"]
    steps = []

    for i, bag in enumerate(info["bag_list"]):
        bag_id = bag["id"]
        dims = bag["size"]
        weight = bag["weight"]
        dim_sum = sum(dims)

        correct_oversize_fee = gt_info["oversize"][i]
        correct_overweight_fee = gt_info["overweight"][i]
        correct_base_fee = gt_info["base"][i]

        # ── ELEMENTARY A: Pure arithmetic ─────────────────────
        steps.append({
            "step_id": f"bag{bag_id}_EA",
            "bag_id": bag_id,
            "type": "elementary",
            "description": "Compute dimension sum",
            "system": "You are a calculator. Answer with just the number.",
            "user": f"What is {dims[0]} + {dims[1]} + {dims[2]}? Answer with just the number.",
            "correct_answer": str(dim_sum),
        })

        # ── ELEMENTARY B: Pure comparison ──────────────────────
        expected_oversize = "OVERSIZE" if dim_sum > 62 else "NOT OVERSIZE"
        steps.append({
            "step_id": f"bag{bag_id}_EB",
            "bag_id": bag_id,
            "type": "elementary",
            "description": "Compare sum to threshold",
            "system": "Answer with exactly OVERSIZE or NOT OVERSIZE.",
            "user": f"A bag has total dimensions of {dim_sum} inches. The oversize threshold is 62 inches. Is this bag OVERSIZE or NOT OVERSIZE?",
            "correct_answer": expected_oversize,
        })

        expected_overweight = "OVERWEIGHT" if weight > 50 else "NOT OVERWEIGHT"
        steps.append({
            "step_id": f"bag{bag_id}_EC",
            "bag_id": bag_id,
            "type": "elementary",
            "description": "Compare weight to threshold",
            "system": "Answer with exactly OVERWEIGHT or NOT OVERWEIGHT.",
            "user": f"A bag weighs {weight} lbs. The overweight threshold is 50 lbs. Is this bag OVERWEIGHT or NOT OVERWEIGHT?",
            "correct_answer": expected_overweight,
        })

        # ── ELEMENTARY C: Pure table read (bracket given) ──────
        if dim_sum > 62:
            if dim_sum <= 65:
                bracket = "Over 62 inches to 65 inches"
            else:
                bracket = "Over 65 inches to 115 inches"
            steps.append({
                "step_id": f"bag{bag_id}_ED",
                "bag_id": bag_id,
                "type": "elementary",
                "description": "Read oversize fee from table given bracket",
                "system": "You are an airline fee calculator. Answer with just the dollar amount.",
                "user": (
                    f"Route: US domestic. Class: {gt_info['customer_class']}.\n"
                    f"Bag is in bracket: {bracket}.\n\n"
                    f"OVERSIZE FEE TABLE:\n"
                    f"Between U.S., Puerto Rico, U.S. Virgin Islands and Canada:\n"
                    f"- Over 62 inches to 65 inches: $30\n"
                    f"- Over 65 inches to 115 inches: $200\n\n"
                    f"What is the oversize fee? Answer with just the dollar amount."
                ),
                "correct_answer": f"${correct_oversize_fee}",
            })

        if weight > 50:
            if weight <= 53:
                wbracket = "Over 50 lbs to 53 lbs"
            elif weight <= 70:
                wbracket = "Over 53 lbs to 70 lbs"
            else:
                wbracket = "Over 70 lbs to 100 lbs"
            steps.append({
                "step_id": f"bag{bag_id}_EE",
                "bag_id": bag_id,
                "type": "elementary",
                "description": "Read overweight fee from table given bracket",
                "system": "You are an airline fee calculator. Answer with just the dollar amount.",
                "user": (
                    f"Route: US domestic. Class: {gt_info['customer_class']}.\n"
                    f"Bag is in bracket: {wbracket}.\n\n"
                    f"OVERWEIGHT FEE TABLE:\n"
                    f"- Over 50 lbs to 53 lbs: $30\n"
                    f"- Over 53 lbs to 70 lbs: $100\n"
                    f"- Over 70 lbs to 100 lbs: $200\n\n"
                    f"What is the overweight fee? Answer with just the dollar amount."
                ),
                "correct_answer": f"${correct_overweight_fee}",
            })

        # ── BOUNDARY: Raw value → derive bracket → lookup fee ──
        if dim_sum > 62:
            steps.append({
                "step_id": f"bag{bag_id}_B1",
                "bag_id": bag_id,
                "type": "boundary",
                "description": "Raw dimensions → oversize fee (must derive bracket + lookup)",
                "system": "You are an airline fee calculator. Answer with just the dollar amount.",
                "user": (
                    f"Route: US domestic. Class: {gt_info['customer_class']}.\n"
                    f"Bag dimensions: {dims[0]} x {dims[1]} x {dims[2]} inches.\n\n"
                    f"FEE TABLE:\n"
                    f"{REFERENCE_RULES}\n\n"
                    f"What is the oversize fee for this bag? Answer with just the dollar amount."
                ),
                "correct_answer": f"${correct_oversize_fee}",
            })

        if weight > 50:
            steps.append({
                "step_id": f"bag{bag_id}_B2",
                "bag_id": bag_id,
                "type": "boundary",
                "description": "Raw weight → overweight fee (must derive bracket + lookup)",
                "system": "You are an airline fee calculator. Answer with just the dollar amount.",
                "user": (
                    f"Route: US domestic. Class: {gt_info['customer_class']}.\n"
                    f"Bag weight: {weight} lbs.\n\n"
                    f"FEE TABLE:\n"
                    f"{REFERENCE_RULES}\n\n"
                    f"What is the overweight fee for this bag? Answer with just the dollar amount."
                ),
                "correct_answer": f"${correct_overweight_fee}",
            })

    return steps

def extract_answer(response, step_type):
    response = response.strip()
    for keyword in ["NOT OVERSIZE", "NOT OVERWEIGHT", "OVERSIZE", "OVERWEIGHT"]:
        if keyword in response.upper():
            return keyword
    match = re.search(r'\$(\d+)', response)
    if match:
        return f"${match.group(1)}"
    nums = re.findall(r'\b\d+\b', response)
    if nums:
        return nums[0]
    return response.strip()

# Test on problem 0
gt, gt_info = compute_gt(problems[0]["info"])
steps = generate_steps(problems[0], gt_info)
elem = [s for s in steps if s['type'] == 'elementary']
bound = [s for s in steps if s['type'] == 'boundary']
print(f"Problem 0: {len(steps)} steps — {len(elem)} elementary, {len(bound)} boundary")
for s in steps:
    print(f"  {s['step_id']} ({s['type']}): {s['description']} → correct={s['correct_answer']}")

Problem 0: 31 steps — 23 elementary, 8 boundary
  bag1_EA (elementary): Compute dimension sum → correct=41
  bag1_EB (elementary): Compare sum to threshold → correct=NOT OVERSIZE
  bag1_EC (elementary): Compare weight to threshold → correct=NOT OVERWEIGHT
  bag2_EA (elementary): Compute dimension sum → correct=86
  bag2_EB (elementary): Compare sum to threshold → correct=OVERSIZE
  bag2_EC (elementary): Compare weight to threshold → correct=OVERWEIGHT
  bag2_ED (elementary): Read oversize fee from table given bracket → correct=$200
  bag2_EE (elementary): Read overweight fee from table given bracket → correct=$100
  bag2_B1 (boundary): Raw dimensions → oversize fee (must derive bracket + lookup) → correct=$200
  bag2_B2 (boundary): Raw weight → overweight fee (must derive bracket + lookup) → correct=$100
  bag3_EA (elementary): Compute dimension sum → correct=64
  bag3_EB (elementary): Compare sum to threshold → correct=OVERSIZE
  bag3_EC (elementary): Compare weight to threshold → cor

In [ ]:
def run_steps(steps):
    results = []
    for step in steps:
        response = query(step["system"], step["user"], max_new_tokens=100)
        predicted = extract_answer(response, step["type"])
        correct = predicted.strip().upper() == step["correct_answer"].strip().upper()
        results.append({
            "step_id": step["step_id"],
            "type": step["type"],
            "description": step["description"],
            "correct_answer": step["correct_answer"],
            "predicted": predicted,
            "correct": correct,
        })
        print(f"{step['step_id']} ({step['type']}): expected={step['correct_answer']} predicted={predicted} {'✓' if correct else '✗'}")
    return results

print("=== PROBLEM 0 — ALL STEPS ===\n")
gt, gt_info = compute_gt(problems[0]["info"])
steps = generate_steps(problems[0], gt_info)
results_p0 = run_steps(steps)

elem_correct = sum(1 for r in results_p0 if r["type"] == "elementary" and r["correct"])
elem_total = sum(1 for r in results_p0 if r["type"] == "elementary")
bound_correct = sum(1 for r in results_p0 if r["type"] == "boundary" and r["correct"])
bound_total = sum(1 for r in results_p0 if r["type"] == "boundary")

print(f"\n=== SUMMARY ===")
print(f"Elementary: {elem_correct}/{elem_total} correct ({elem_correct/elem_total*100:.1f}%)")
print(f"Boundary:   {bound_correct}/{bound_total} correct ({bound_correct/bound_total*100:.1f}%)")

=== PROBLEM 0 — ALL STEPS ===

bag1_EA (elementary): expected=41 predicted=41 ✓
bag1_EB (elementary): expected=NOT OVERSIZE predicted=NOT OVERSIZE ✓
bag1_EC (elementary): expected=NOT OVERWEIGHT predicted=NOT OVERWEIGHT ✓
bag2_EA (elementary): expected=86 predicted=86 ✓
bag2_EB (elementary): expected=OVERSIZE predicted=OVERSIZE ✓
bag2_EC (elementary): expected=OVERWEIGHT predicted=OVERWEIGHT ✓
bag2_ED (elementary): expected=$200 predicted=$200 ✓
bag2_EE (elementary): expected=$100 predicted=$100 ✓
bag2_B1 (boundary): expected=$200 predicted=$30 ✗
bag2_B2 (boundary): expected=$100 predicted=$100 ✓
bag3_EA (elementary): expected=64 predicted=64 ✓
bag3_EB (elementary): expected=OVERSIZE predicted=OVERSIZE ✓
bag3_EC (elementary): expected=OVERWEIGHT predicted=OVERWEIGHT ✓
bag3_ED (elementary): expected=$30 predicted=$30 ✓
bag3_EE (elementary): expected=$30 predicted=$30 ✓
bag3_B1 (boundary): expected=$30 predicted=$30 ✓
bag3_B2 (boundary): expected=$30 predicted=$30 ✓
bag4_EA (elementary):

In [ ]:
print("\n=== PROBLEMS 1 AND 2 ===\n")

all_results = results_p0

for i in range(1, 3):
    print(f"\n--- Problem {i} ---")
    gt, gt_info = compute_gt(problems[i]["info"])
    steps = generate_steps(problems[i], gt_info)
    results = run_steps(steps)
    all_results.extend(results)

elem_correct = sum(1 for r in all_results if r["type"] == "elementary" and r["correct"])
elem_total = sum(1 for r in all_results if r["type"] == "elementary")
bound_correct = sum(1 for r in all_results if r["type"] == "boundary" and r["correct"])
bound_total = sum(1 for r in all_results if r["type"] == "boundary")

print(f"\n=== COMBINED SUMMARY (3 problems) ===")
print(f"Elementary: {elem_correct}/{elem_total} correct ({elem_correct/elem_total*100:.1f}%)")
print(f"Boundary:   {bound_correct}/{bound_total} correct ({bound_correct/bound_total*100:.1f}%)")


=== PROBLEMS 1 AND 2 ===


--- Problem 1 ---
bag1_EA (elementary): expected=37 predicted=37 ✓
bag1_EB (elementary): expected=NOT OVERSIZE predicted=NOT OVERSIZE ✓
bag1_EC (elementary): expected=NOT OVERWEIGHT predicted=NOT OVERWEIGHT ✓
bag2_EA (elementary): expected=77 predicted=77 ✓
bag2_EB (elementary): expected=OVERSIZE predicted=OVERSIZE ✓
bag2_EC (elementary): expected=OVERWEIGHT predicted=OVERWEIGHT ✓
bag2_ED (elementary): expected=$200 predicted=$200 ✓
bag2_EE (elementary): expected=$200 predicted=$200 ✓
bag2_B1 (boundary): expected=$200 predicted=$30 ✗
bag2_B2 (boundary): expected=$200 predicted=$100 ✗
bag3_EA (elementary): expected=80 predicted=80 ✓
bag3_EB (elementary): expected=OVERSIZE predicted=OVERSIZE ✓
bag3_EC (elementary): expected=OVERWEIGHT predicted=OVERWEIGHT ✓
bag3_ED (elementary): expected=$200 predicted=$200 ✓
bag3_EE (elementary): expected=$200 predicted=$200 ✓
bag3_B1 (boundary): expected=$200 predicted=$30 ✗
bag3_B2 (boundary): expected=$200 predicted=$100 ✗

In [16]:
def generate_steps(problem, gt_info):
    info = problem["info"]
    steps = []

    for i, bag in enumerate(info["bag_list"]):
        bag_id = bag["id"]
        dims = bag["size"]
        weight = bag["weight"]
        dim_sum = sum(dims)

        correct_oversize_fee = gt_info["oversize"][i]
        correct_overweight_fee = gt_info["overweight"][i]
        correct_base_fee = gt_info["base"][i]

        # ── ELEMENTARY: pure arithmetic and comparison only ────

        steps.append({
            "step_id": f"bag{bag_id}_EA",
            "bag_id": bag_id,
            "type": "elementary",
            "description": "Compute dimension sum",
            "system": "You are a calculator. Answer with just the number.",
            "user": f"What is {dims[0]} + {dims[1]} + {dims[2]}? Answer with just the number.",
            "correct_answer": str(dim_sum),
        })

        expected_oversize = "OVERSIZE" if dim_sum > 62 else "NOT OVERSIZE"
        steps.append({
            "step_id": f"bag{bag_id}_EB",
            "bag_id": bag_id,
            "type": "elementary",
            "description": "Compare sum to oversize threshold",
            "system": "Answer with exactly OVERSIZE or NOT OVERSIZE.",
            "user": (
                f"A bag has total dimensions of {dim_sum} inches. "
                f"The oversize threshold is 62 inches. "
                f"Is this bag OVERSIZE or NOT OVERSIZE?"
            ),
            "correct_answer": expected_oversize,
        })

        expected_overweight = "OVERWEIGHT" if weight > 50 else "NOT OVERWEIGHT"
        steps.append({
            "step_id": f"bag{bag_id}_EC",
            "bag_id": bag_id,
            "type": "elementary",
            "description": "Compare weight to overweight threshold",
            "system": "Answer with exactly OVERWEIGHT or NOT OVERWEIGHT.",
            "user": (
                f"A bag weighs {weight} lbs. "
                f"The overweight threshold is 50 lbs. "
                f"Is this bag OVERWEIGHT or NOT OVERWEIGHT?"
            ),
            "correct_answer": expected_overweight,
        })

        # ── ED: table read oversize — elementary if fee>0, boundary if fee=0
        if dim_sum > 62:
            if dim_sum <= 65:
                bracket = "Over 62 inches to 65 inches"
            else:
                bracket = "Over 65 inches to 115 inches"
            steps.append({
                "step_id": f"bag{bag_id}_ED",
                "bag_id": bag_id,
                "type": "elementary" if correct_oversize_fee > 0 else "boundary",
                "description": "Read oversize fee from table given bracket"
                               + (" (complimentary rule applies → boundary)" if correct_oversize_fee == 0 else ""),
                "system": "You are an airline fee calculator. Answer with just the dollar amount.",
                "user": (
                    f"Route: US domestic. Class: {gt_info['customer_class']}.\n"
                    f"Bag is in bracket: {bracket}.\n\n"
                    f"OVERSIZE FEE TABLE:\n"
                    f"Between U.S., Puerto Rico, U.S. Virgin Islands and Canada:\n"
                    f"- Over 62 inches to 65 inches: $30\n"
                    f"- Over 65 inches to 115 inches: $200\n\n"
                    f"What is the oversize fee for this passenger? "
                    f"Answer with just the dollar amount."
                ),
                "correct_answer": f"${correct_oversize_fee}",
            })

        # ── EE: table read overweight — elementary if fee>0, boundary if fee=0
        if weight > 50:
            if weight <= 53:
                wbracket = "Over 50 lbs to 53 lbs"
            elif weight <= 70:
                wbracket = "Over 53 lbs to 70 lbs"
            else:
                wbracket = "Over 70 lbs to 100 lbs"
            steps.append({
                "step_id": f"bag{bag_id}_EE",
                "bag_id": bag_id,
                "type": "elementary" if correct_overweight_fee > 0 else "boundary",
                "description": "Read overweight fee from table given bracket"
                               + (" (complimentary rule applies → boundary)" if correct_overweight_fee == 0 else ""),
                "system": "You are an airline fee calculator. Answer with just the dollar amount.",
                "user": (
                    f"Route: US domestic. Class: {gt_info['customer_class']}.\n"
                    f"Bag is in bracket: {wbracket}.\n\n"
                    f"OVERWEIGHT FEE TABLE:\n"
                    f"- Over 50 lbs to 53 lbs: $30\n"
                    f"- Over 53 lbs to 70 lbs: $100\n"
                    f"- Over 70 lbs to 100 lbs: $200\n\n"
                    f"What is the overweight fee for this passenger? "
                    f"Answer with just the dollar amount."
                ),
                "correct_answer": f"${correct_overweight_fee}",
            })

        # ── BOUNDARY: everything requiring rule knowledge ──────

        steps.append({
            "step_id": f"bag{bag_id}_B0",
            "bag_id": bag_id,
            "type": "boundary",
            "description": "Is bag complimentary? (class + bag number + route rule)",
            "system": "Answer with exactly YES or NO.",
            "user": (
                f"Passenger class: {gt_info['customer_class']}\n"
                f"Route: {gt_info['routine']} domestic\n"
                f"This is bag number {bag_id} "
                f"(the {['1st','2nd','3rd','4th','5th'][bag_id-1]} checked bag).\n\n"
                f"FEE TABLE:\n{REFERENCE_RULES}\n\n"
                f"Is this bag complimentary (free) for this passenger? "
                f"Answer with exactly YES or NO."
            ),
            "correct_answer": "YES" if correct_base_fee == 0 else "NO",
        })

        if dim_sum > 62:
            steps.append({
                "step_id": f"bag{bag_id}_B1",
                "bag_id": bag_id,
                "type": "boundary",
                "description": "Raw dimensions → oversize fee (derive bracket + lookup)",
                "system": "Answer with just the dollar amount.",
                "user": (
                    f"Route: US domestic. Class: {gt_info['customer_class']}.\n"
                    f"Bag dimensions: {dims[0]} x {dims[1]} x {dims[2]} inches.\n\n"
                    f"FEE TABLE:\n{REFERENCE_RULES}\n\n"
                    f"What is the oversize fee for this bag? "
                    f"Answer with just the dollar amount."
                ),
                "correct_answer": f"${correct_oversize_fee}",
            })

        if weight > 50:
            steps.append({
                "step_id": f"bag{bag_id}_B2",
                "bag_id": bag_id,
                "type": "boundary",
                "description": "Raw weight → overweight fee (derive bracket + lookup)",
                "system": "Answer with just the dollar amount.",
                "user": (
                    f"Route: US domestic. Class: {gt_info['customer_class']}.\n"
                    f"Bag weight: {weight} lbs.\n\n"
                    f"FEE TABLE:\n{REFERENCE_RULES}\n\n"
                    f"What is the overweight fee for this bag? "
                    f"Answer with just the dollar amount."
                ),
                "correct_answer": f"${correct_overweight_fee}",
            })

    return steps

def extract_answer(response, step_type):
    response = response.strip()
    for keyword in ["NOT OVERSIZE", "NOT OVERWEIGHT", "OVERSIZE", "OVERWEIGHT"]:
        if keyword in response.upper():
            return keyword
    if response.upper().startswith("YES"):
        return "YES"
    if response.upper().startswith("NO"):
        return "NO"
    match = re.search(r'\$(\d+)', response)
    if match:
        return f"${match.group(1)}"
    nums = re.findall(r'\b\d+\b', response)
    if nums:
        return nums[0]
    return response.strip()

# Verify on problem 0
gt, gt_info = compute_gt(problems[0]["info"])
steps = generate_steps(problems[0], gt_info)
elem = [s for s in steps if s['type'] == 'elementary']
bound = [s for s in steps if s['type'] == 'boundary']
print(f"Problem 0: {len(steps)} steps — {len(elem)} elementary, {len(bound)} boundary")
for s in steps:
    print(f"  {s['step_id']} ({s['type']}): {s['description']} → correct={s['correct_answer']}")

Problem 0: 36 steps — 23 elementary, 13 boundary
  bag1_EA (elementary): Compute dimension sum → correct=41
  bag1_EB (elementary): Compare sum to oversize threshold → correct=NOT OVERSIZE
  bag1_EC (elementary): Compare weight to overweight threshold → correct=NOT OVERWEIGHT
  bag1_B0 (boundary): Is bag complimentary? (class + bag number + route rule) → correct=NO
  bag2_EA (elementary): Compute dimension sum → correct=86
  bag2_EB (elementary): Compare sum to oversize threshold → correct=OVERSIZE
  bag2_EC (elementary): Compare weight to overweight threshold → correct=OVERWEIGHT
  bag2_ED (elementary): Read oversize fee from table given bracket → correct=$200
  bag2_EE (elementary): Read overweight fee from table given bracket → correct=$100
  bag2_B0 (boundary): Is bag complimentary? (class + bag number + route rule) → correct=NO
  bag2_B1 (boundary): Raw dimensions → oversize fee (derive bracket + lookup) → correct=$200
  bag2_B2 (boundary): Raw weight → overweight fee (derive brac

In [19]:
import os, time

# Fresh start
if os.path.exists("/content/airline_step_results.json"):
    os.remove("/content/airline_step_results.json")
    print("Old results deleted — starting fresh.")

RESULTS_FILE = "/content/airline_step_results.json"
all_results = []
done_problems = set()

for prob_idx in range(60):
    print(f"\n--- Problem {prob_idx+1}/60 ---")
    gt, gt_info = compute_gt(problems[prob_idx]["info"])
    steps = generate_steps(problems[prob_idx], gt_info)

    prob_results = []
    for step in steps:
        response = query(step["system"], step["user"], max_new_tokens=100)
        predicted = extract_answer(response, step["type"])
        correct = predicted.strip().upper() == step["correct_answer"].strip().upper()
        prob_results.append({
            "problem_idx": prob_idx,
            "step_id": step["step_id"],
            "type": step["type"],
            "description": step["description"],
            "correct_answer": step["correct_answer"],
            "predicted": predicted,
            "correct": correct,
            "customer_class": gt_info["customer_class"],
        })
        print(f"  {step['step_id']} ({step['type']}): expected={step['correct_answer']} predicted={predicted} {'✓' if correct else '✗'}")

    all_results.extend(prob_results)
    done_problems.add(prob_idx)

    with open(RESULTS_FILE, "w") as f:
        json.dump(all_results, f, indent=2)
    print(f"  [Saved problem {prob_idx+1}]")

# ── FINAL SUMMARY ─────────────────────────────────────────────
print("\n\n=== FINAL SUMMARY — ALL 60 PROBLEMS ===\n")

elem = [r for r in all_results if r["type"] == "elementary"]
bound = [r for r in all_results if r["type"] == "boundary"]

elem_correct = sum(1 for r in elem if r["correct"])
bound_correct = sum(1 for r in bound if r["correct"])

print(f"Total steps: {len(all_results)} ({len(elem)} elementary, {len(bound)} boundary)")
print(f"\nELEMENTARY: {elem_correct}/{len(elem)} correct ({elem_correct/len(elem)*100:.1f}%)")
print(f"BOUNDARY:   {bound_correct}/{len(bound)} correct ({bound_correct/len(bound)*100:.1f}%)")
print()

print("Elementary breakdown:")
for code, desc in [("EA","Compute dimension sum"), ("EB","Compare dim to threshold"),
                   ("EC","Compare weight to threshold"), ("ED","Read oversize fee (bracket given)"),
                   ("EE","Read overweight fee (bracket given)")]:
    s = [r for r in elem if code in r["step_id"]]
    if s:
        c = sum(1 for r in s if r["correct"])
        print(f"  {code} ({desc}): {c}/{len(s)} ({c/len(s)*100:.1f}%)")

print()
print("Boundary breakdown:")
for code, desc in [("B0","Complimentary bag rule"), ("B1","Raw dim → oversize fee"),
                   ("B2","Raw weight → overweight fee"), ("ED","Oversize fee (complimentary=0)"),
                   ("EE","Overweight fee (complimentary=0)")]:
    s = [r for r in bound if code in r["step_id"]]
    if s:
        c = sum(1 for r in s if r["correct"])
        print(f"  {code} ({desc}): {c}/{len(s)} ({c/len(s)*100:.1f}%)")

from google.colab import files
files.download(RESULTS_FILE)


--- Problem 1/60 ---
  bag1_EA (elementary): expected=41 predicted=41 ✓
  bag1_EB (elementary): expected=NOT OVERSIZE predicted=NOT OVERSIZE ✓
  bag1_EC (elementary): expected=NOT OVERWEIGHT predicted=NOT OVERWEIGHT ✓
  bag1_B0 (boundary): expected=NO predicted=NO ✓
  bag2_EA (elementary): expected=86 predicted=86 ✓
  bag2_EB (elementary): expected=OVERSIZE predicted=OVERSIZE ✓
  bag2_EC (elementary): expected=OVERWEIGHT predicted=OVERWEIGHT ✓
  bag2_ED (elementary): expected=$200 predicted=$200 ✓
  bag2_EE (elementary): expected=$100 predicted=$100 ✓
  bag2_B0 (boundary): expected=NO predicted=NO ✓
  bag2_B1 (boundary): expected=$200 predicted=$30 ✗
  bag2_B2 (boundary): expected=$100 predicted=$100 ✓
  bag3_EA (elementary): expected=64 predicted=64 ✓
  bag3_EB (elementary): expected=OVERSIZE predicted=OVERSIZE ✓
  bag3_EC (elementary): expected=OVERWEIGHT predicted=OVERWEIGHT ✓
  bag3_ED (elementary): expected=$30 predicted=$30 ✓
  bag3_EE (elementary): expected=$30 predicted=$30 ✓


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>